# 07_model_randomforest.ipynb

## 목차

### 1. 환경 세팅 및 데이터 로드
### 2. 데이터셋 분할
### 3. NaN 처리 (Random Forest는 NaN 불가)
### 4. 베이스라인 모델 학습
### 5. 하이퍼파라미터 튜닝
### 6. 최종 모델 학습 및 평가
### 7. 피처 중요도 분석
### 8. 결과 저장

## 1. 환경 세팅 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import time, os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, roc_curve, classification_report

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.4f}'.format
print('라이브러리 임포트 완료')

In [ ]:
df = pd.read_csv('./data/hnm_churn_processed.csv')
print(f'데이터 shape: {df.shape}')
print(f'결측치 합계: {df.isnull().sum().sum()}')

## 2. 데이터셋 분할

In [ ]:
X = df.drop(columns=['churn'])
y = df['churn']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')
print(f'y_train 이탈률: {y_train.mean():.4f}')

## 3. NaN 처리

Random Forest는 NaN을 처리하지 못합니다.
`avg_purchase_interval`, `std_purchase_interval`의 NaN을 median으로 대치합니다.

In [ ]:
# avg/std_purchase_interval NaN → median 대치
# (03_preprocessing 주석 처리된 코드와 동일한 로직)
for col in ['avg_purchase_interval', 'std_purchase_interval']:
    if col in X_train.columns:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_val[col]   = X_val[col].fillna(median_val)
        print(f'{col} NaN → {median_val:.2f} 대치 완료')

print(f'처리 후 결측치: {X_train.isnull().sum().sum()}')

## 4. 베이스라인 모델 학습

In [ ]:
baseline_model = RandomForestClassifier(
    random_state=42, n_jobs=-1
)

start = time.time()
baseline_model.fit(X_train, y_train)
baseline_time = time.time() - start

baseline_pred  = baseline_model.predict(X_val)
baseline_proba = baseline_model.predict_proba(X_val)[:, 1]

print('--- Random Forest 베이스라인 성능 ---')
print(f'Accuracy : {accuracy_score(y_val, baseline_pred):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_val, baseline_proba):.4f}')
print(f'F1-Score : {f1_score(y_val, baseline_pred):.4f}')
print(f'학습 시간 : {baseline_time:.1f}초')

## 5. 하이퍼파라미터 튜닝

In [ ]:
param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [10, 20, None],
    'min_samples_split': [2, 5],
    'max_features'     : ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    estimator=rf_base, param_grid=param_grid,
    scoring='roc_auc', cv=3, n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print('--- 최적 파라미터 ---')
print(grid_search.best_params_)
print(f'CV 최고 AUC: {grid_search.best_score_:.4f}')

## 6. 최종 모델 학습 및 평가

In [ ]:
best_params = grid_search.best_params_

final_model = RandomForestClassifier(
    **best_params,
    random_state=42, n_jobs=-1
)

start = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - start
print(f'최종 학습 완료 ({train_time:.1f}초)')

In [ ]:
y_pred  = final_model.predict(X_val)
y_proba = final_model.predict_proba(X_val)[:, 1]

final_acc = accuracy_score(y_val, y_pred)
final_auc = roc_auc_score(y_val, y_proba)
final_f1  = f1_score(y_val, y_pred)

print('--- Random Forest 최종 성능 ---')
print(f'Accuracy : {final_acc:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print(f'F1-Score : {final_f1:.4f}')
print(f'학습 시간 : {train_time:.1f}초')
print()
print(classification_report(y_val, y_pred, target_names=['유지(0)', '이탈(1)']))

In [ ]:
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['유지(0)', '이탈(1)'], yticklabels=['유지(0)', '이탈(1)'])
plt.title('Random Forest Confusion Matrix')
plt.ylabel('실제값')
plt.xlabel('예측값')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {final_auc:.4f})', color='purple', lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 7. 피처 중요도 분석

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(x='importance', y='feature', data=importance_df, palette='Purples_r')
plt.title('Random Forest Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

print('--- 피처 중요도 Top 10 ---')
print(importance_df[['feature', 'importance']].head(10).to_string(index=False))

## 8. 결과 저장

In [ ]:
os.makedirs('./results', exist_ok=True)

result = pd.DataFrame([{
    'model'     : 'RandomForest',
    'accuracy'  : round(final_acc, 4),
    'auc_roc'   : round(final_auc, 4),
    'f1_score'  : round(final_f1, 4),
    'train_time': round(train_time, 1)
}])

result.to_csv('./results/rf_result.csv', index=False)
print('결과 저장 완료')
print(result.to_string(index=False))